# Tutorial 1: Basic Python Integration

## Goals

- Introduce DataStructure, Filter, Parameter, DataPath
- Run basic python code to execute a filter
- Visualize a DataArray using MatPlotLib


## Import Statements

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import simplnx as nx
import orientationanalysis as nxor

## Utility Function

We can use this function to print the any warnings and/or errors from the filter and to stop further execution on failure.

In [ ]:
# This is a utility function for printing warnings/errors from executing a filter.
# Then if there is an error, it will raise an exception.
# See PythonTutorial/nxutility.py for details
from nxutility import check_filter_result

## Executing a filter

In [ ]:
# Create the DataStructure Object
data_structure = nx.DataStructure()
# Create the Filter object
nx_filter = nxor.ReadAngDataFilter()
# Execute the filter
result = nx_filter.execute(
   data_structure=data_structure,
   cell_attribute_matrix_name='Cell Data',
   cell_ensemble_attribute_matrix_name='Cell Ensemble Data',
   output_image_geometry_path=nx.DataPath('DataContainer'),
   input_file='Data/Small_IN100/Slice_1.ang',
)
check_filter_result(nx_filter, result)

## Accessing NumPy Arrays

Use the `npview()` function to get a **view** into the DataArray. This is **NOT** a copy 
of the data. 

- Views all access the same underlying chunk of memory
- Copies will allocate a new chunk of memory and copy the data into that

[NumPy copy vs view docs](https://numpy.org/doc/stable/user/basics.copies.html)

The data we are reading will have some erroneous negative values for the "Confidence Index" array.

We are going to fix this issue by constraining all values to be at least 0 using [numpy.clip](https://numpy.org/doc/stable/reference/generated/numpy.clip.html).

In [ ]:
# Get a numpy view of the Confidence Index Array and ensure all dimensions are
# are > 1
ci_view: np.ndarray = data_structure['DataContainer/Cell Data/Confidence Index'].npview()

# Ensure all values are at least 0
np.clip(ci_view, a_min=0, a_max=None, out=ci_view)

## Plotting

Let's use Matplotlib to render the EBSD Confidence Index array.

Before plotting we must use [`squeeze()`](https://numpy.org/doc/stable/reference/generated/numpy.squeeze.html).
DREAM3D will have some dimensions that are 1.

- For example, a single component array such as "Confidence Index"
- A single layer of an Image Geometry with Z dimension = 1
- For a typical 3D array, the dimensions will be of the form (Z, Y, X, N) where N is the number of components per voxel

Matplotlib accepts array-like data with a shape of `(M, N)`. `squeeze()` removes axes of length one allowing us to plot our data.

In this, case our Z dimension is 1 so our data is 2D. Our component dimension is also 1 and doesn't affect the data size.

In [ ]:
print(f'DREAM3D Dimensions: {ci_view.shape}')

ci_view = ci_view.squeeze()

print(f'NumPy Dimensions: {ci_view.shape}')

In [ ]:
# Show the result
image_plot = plt.imshow(ci_view)
colorbar = plt.colorbar(image_plot)
colorbar.set_label('Color Scale')
plt.title('Corrected Confidence Index')
plt.axis('on') # to turn on axes
plt.show()

## Questions?